In [0]:
from pyspark.sql.functions import col, when, lit, round, current_timestamp
bronze_df = spark.table('db_test.bronze.movies')
df2 = bronze_df.withColumn('studio', when(col('studio').isNull(), lit('Not Available')).otherwise(col('studio'))).\
    withColumn('imdb_rating', when(col('imdb_rating')=='NULL', lit('0')).otherwise(col('imdb_rating')))
# display(df2)

df3 = df2.withColumn("release_year", col("release_year").cast("int")) \
             .withColumn("imdb_rating", col("imdb_rating").cast("decimal(10,1)")) \
                 .withColumn("budget", col("budget").cast("double")) \
                     .withColumn("revenue", col("revenue").cast("double"))
# display(df3)
# df3.printSchema()
df4 = df3.withColumn('ingested_on',current_timestamp())\
    .withColumn('source',col("_metadata.file_path"))
display(df4)
df4.write.format('delta').mode('overwrite').saveAsTable('db_test.silver.movies')

In [0]:
spark.table("db_test.silver.movies").describe().show()

In [0]:
from pyspark.sql import Row

new_data = [
    Row(title="Inception", industry="HOLLYWOOD", release_year=2010, imdb_rating=9.0, studio="Warner Bros"),
    Row(title="Interstellar", industry="HOLLYWOOD", release_year=2014, imdb_rating=9.1, studio="Warner Bros")  # updated rating
]

df_new = spark.createDataFrame(new_data)
df_new.show()

# **MERGE**

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "db_test.silver.movies")

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "db_test.silver.movies")

delta_table.alias("target").merge(
    df_new.alias("source"),
    "target.title = source.title"
).whenMatchedUpdate(set={
    "industry": "source.industry",
    "release_year": "source.release_year",
    "imdb_rating": "source.imdb_rating",
    "studio": "source.studio"
    # not touching budget, revenue, etc.
}) \
.whenNotMatchedInsert(values={
    "title": "source.title",
    "industry": "source.industry",
    "release_year": "source.release_year",
    "imdb_rating": "source.imdb_rating",
    "studio": "source.studio",
    "budget": "NULL",
    "revenue": "NULL",
    "unit": "NULL",
    "currency": "NULL",
    "language": "NULL",
    "ingested_on": "current_timestamp()",
    "source": "'incremental_load'"
}) \
.execute()
spark.table("db_test.silver.movies").show()

In [0]:
spark.table("db_test.silver.movies") \
    .filter("title in ('Inception', 'Interstellar')") \
    .show()

**REVENUE **TABLE****

In [0]:
from pyspark.sql.functions import sum

gold_industry = spark.table("db_test.silver.movies") \
    .groupBy("industry") \
    .agg(sum("revenue").alias("total_revenue"))
display(gold_industry)
gold_industry.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_test.gold.revenue_by_industry")

**Top Rated Movies**

In [0]:
from pyspark.sql.functions import desc

gold_top_movies = spark.table("db_test.silver.movies") \
    .orderBy(desc("imdb_rating")) \
    .limit(10)
display(gold_top_movies)
gold_top_movies.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_test.gold.top_movies")

## **Revenue by Year**

In [0]:
gold_year = spark.table("db_test.silver.movies") \
    .groupBy("release_year") \
    .agg(sum("revenue").alias("total_revenue"))
display(gold_year)
gold_year.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_test.gold.revenue_by_year")

In [0]:
%sql
OPTIMIZE db_test.silver.movies
ZORDER BY (title);

# **Commit**